# Activity 4: Binary to Decimal Conversion

Activity 4 uses a separate project folder and introduces binary to decimal conversion.

## Opening the Activity 4 Project

1. Navigate to the `Introductory Workshop/Activities/FPGA_Activity_4` folder in GitHub and download it.

2. Open Intel Quartus Prime (if not already open).

3. In Quartus, go to **File → Open Project**.

4. Browse to the `FPGA_Activity_4` folder and open the project file (`.qpf` extension).

## Setting the Top-Level Entity

Once the project is open, you need to set the correct top-level entity:

1. In Quartus, go to **Project → Set as Top-Level Entity**.

2. Select activity_4 as the top-level file.

3. Alternatively, you can right-click on the file in the Project Navigator and select **Set as Top-Level Entity**.

## From Digital to Programmed Logic

### Function

A function is a reusable block of code that executes a specific computation and returns a value. Verilog functions are like regular programming languages.

```verilog
function automatic [6:0] to_seven_segment;
    input [3:0] digit;
    case (digit)
        4'd0: to_seven_segment = 7'b1000000;
        4'd1: to_seven_segment = 7'b1111001;
        4'd2: to_seven_segment = 7'b0100100;
        4'd3: to_seven_segment = 7'b0110000;
        4'd4: to_seven_segment = 7'b0011001;
        4'd5: to_seven_segment = 7'b0010010;
        4'd6: to_seven_segment = 7'b0000010;
        4'd7: to_seven_segment = 7'b1111000;
        4'd8: to_seven_segment = 7'b0000000;
        4'd9: to_seven_segment = 7'b0010000;
        default: to_seven_segment = 7'b1111111; // blank
    endcase
endfunction
```

### Case Statement

A case statement is like a multiple-choice decision. It checks the value of a signal or variable and executes different code depending on the value.

```verilog
case (signal)
    2'b00: // do something when signal is 00
    2'b01: // do something when signal is 01
    2'b10: // do something when signal is 10
    2'b11: // do something when signal is 11
    default: // do something for any other value
endcase
```

### Variables

In Verilog, `logic` signals are used to represent connections and model hardware wires. Inside `always` blocks and functions, procedural variables update their values immediately within the block. Use `logic` signals when you want to communicate between blocks or drive outputs, and use local variables (declared as `int` or automatic variables) when you need quick updates within a single `always` block or function.

In SystemVerilog, the `=` operator is used for blocking (immediate) assignment inside `always_comb` blocks and functions, while `<=` is used for non-blocking (scheduled) assignment inside `always_ff` blocks (sequential logic). So, use `=` for quick, local calculations in combinational logic, and `<=` when updating signals in sequential logic.

## Example Code

This sample code shows the usage of functions and case statements. Again, you can use the syntax of this file as a reference for the following activity.

```verilog
module SegExample(
    input  [1:0] SW,
    output logic [6:0] HEX0  // 7-seg display (active-low)
);

    // Signal declarations
    logic result;

    // Function: Convert digit (0-9) to 7-segment encoding
    function automatic [6:0] to_seven_segment;
        input [3:0] digit;
        case (digit)
            4'd0: to_seven_segment = 7'b1000000;  // 7-seg for 0
            4'd1: to_seven_segment = 7'b1111001;  // 7-seg for 1
            default: to_seven_segment = 7'b1111111; // blank (all off)
        endcase
    endfunction

    assign result = SW[0] & SW[1];

    // Use function to choose display value
    always_comb begin
        if (SW[0] == 1'b0 && SW[1] == 1'b0)
            HEX0 = to_seven_segment(4'dx); // undefined -> blank
        else if (result == 1'b1)
            HEX0 = to_seven_segment(4'd1);
        else
            HEX0 = to_seven_segment(4'd0);
    end

endmodule
```

## Activity Instructions

The goal of this activity is to input a 10-bit signed number in binary, using the switches on the FPGA, and output the number in decimal form on the seven-segment display.

### Generating Digits

Create a function which takes an integer type input variable, returns a `logic [6:0]` type, and matches it to the seven-digit binary number that will drive the seven-segment display correctly.

To do this, use a case statement, which assigns the correct seven-digit binary number to a variable that is returned by the function. Refer to the examples above for syntax.

### Generating Sign

Create a function that takes a `logic` type input, checks whether it is high or low, and returns a `logic [6:0]` type.

If the input is high, return a seven-digit binary number that will display a negative sign. Otherwise, return a value that will result in a blank seven-segment display.

### Handling Inputs

An `always_comb` block that is triggered by the switches has been created in your file.

We have given you most the body code for this block. Revise it and ensure you understand the datatype of each variable.

#### Determining the Sign Bit

This part of the code checks whether the most significant bit, the sign bit, is active or not. If it is, it executes the code which converts the binary input from the first 9 switches to decimal and subtracts 512.

Otherwise, it just converts from binary to decimal. Then, the value is assigned to the signal which carries the signed value, which you defined earlier.

```verilog
// convert input (signed 9-bit number with SW[9] as sign indicator)
if (SW[9] == 1'b1)
    temp_signed = int'(unsigned'(SW[8:0])) - 512;
else
    temp_signed = int'(unsigned'(SW[8:0]));
```

This next part will determine whether the value of the input is greater than or less than 0. Based on that, it will assign either a 0 or 1 to the sign bit signal, and calculate the absolute value of the input, then assign it to the signal which was defined to carry the absolute value.

```verilog
// Get the absolute value and sign
if (temp_signed < 0) begin
    temp_abs = -temp_signed;
    sign_bit = 1'b1;
end else begin
    temp_abs = temp_signed;
    sign_bit = 1'b0;
end
```

#### Decimal Digit Extraction

This section is responsible for decimal digit extraction. It assigns the absolute value from the signal to the variable defined for temporarily storing values.

The hundreds digit is extracted and assigned to the signal defined for the hundreds digit by dividing this value by 100. Then it reassigns the value of temp as the remainder of the absolute value of the input divided by 100.

This value is then divided by 10 to extract the tens place and assigned to the signal defined to carry the tens digit.

The digit to be displayed in the ones place is extracted by using remainder division on the temp variable.

```verilog
// Break into decimal digits
temp = temp_abs;
hundreds = temp / 100;
temp = temp % 100;
tens = temp / 10;
ones = temp % 10;
```

Example: Input=237
```verilog
hundreds = temp / 100; // 237/100 = 2
temp = temp % 100;     // 237 % 100 = 37
tens = temp / 10;      // 37/10 = 3
ones = temp % 10;      // 37 % 10 = 7
```

### Incorporate Helper Functions

Now that the rest of the code is implemented, it's time to use the helper functions we wrote earlier.

#### Displaying the Sign:

Use the sign-generating function (the one that takes a sign bit and returns the corresponding 7-segment code for positive or negative).

Apply this function to the signal that holds the sign bit (e.g., `1'b0` for positive, `1'b1` for negative).

Assign the result to the 7-segment display responsible for showing the sign (HEX3).

#### Displaying the Digits (Hundreds, Tens, Ones):

Use the digit-to-7-segment function, which takes an integer from 0 to 9 and returns the correct 7-bit code to display that digit.

Apply this function to each of the digit signals: hundreds, tens, and ones.

Assign the results to the corresponding 7-segment displays for each place value, hundreds on display HEX2, tens on display HEX1, ones on display HEX0.

This brings us to the end of this activity! You can then upload the file to your FPGA and test whether or not you were successful.

Thank you for participating in the Introductory FPGA workshop!

<div style="display:flex; justify-content:space-between;">
  <span><b>Back: </b> <a href="add_and_display.ipynb">Add and Display</a></span>
  <span></span>
</div>